In [1]:
import hashlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

repo = Path.cwd().parent
sys.path.insert(0, str(repo / "src"))
from cs4406_sem_project.harness import hygiene, memcap, sysinfo

B = repo / "build/adapters"
git = lambda *a: subprocess.run(["git", *a], cwd=repo, capture_output=True, text=True).stdout.strip()
subprocess.run(["make", "adapters"], cwd=repo, check=True, capture_output=True)
assert hygiene.power_source() == "AC Power", f"plug the Mac in first ({hygiene.power_source()})"
assert hygiene.lid_closed() is False, "open the lid first: a closed lid sleeps the Mac despite caffeinate"
run_id = "grid-feasibility-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
raw = repo / "results/raw" / run_id
raw.mkdir(parents=True)
meta = {"run_id": run_id, "git_sha": git("rev-parse", "HEAD"), "dirty": bool(git("status", "--porcelain", "src", "Makefile")),
        "adapters": {b.name: hashlib.sha256(b.read_bytes()).hexdigest() for b in sorted(B.glob("*-adapter"))},
        "sysinfo": sysinfo.snapshot()}
(raw / "meta.json").write_text(json.dumps(meta, indent=1, default=str))
print({k: meta[k] for k in ("run_id", "git_sha", "dirty")})

CAP = 10 * memcap.GIB   # keeps this 16 GiB Mac usable; above it a cell would need most of the machine
LIMIT_S = 3600          # one process: one offline repetition, the online phase's untimed setup, one query
BINARY = {"S0-go": "simplepir-adapter", "S0-cpp": "vsp-adapter", "S1": "vsp-adapter", "S2": "vsp-adapter",
          "M0": "apir-adapter", "M1": "apir-adapter", "S3": "apir-adapter"}

def run_cell(arm, db, rb):
    """One cell, --phase all with one repetition. Returns a row; never raises for the cell's own failure."""
    name = f"{arm}-{db}-{rb}"
    out = raw / f"{name}.jsonl"
    argv = ["caffeinate", "-dims", str(B / BINARY[arm]), "--arm", arm, "--db-bytes", str(db), "--record-bytes", str(rb),
            "--phase", "all", "--reps", "1", "--warmup", "0", "--seed", "1234", "--fault", "none", "--out", str(out)]
    row = {"arm": arm, "db_bytes": db, "record_bytes": rb}
    with hygiene.Watch() as w:
        try:
            r = memcap.run(argv, timeout_s=LIMIT_S, cap_bytes=CAP)
            row.update(exit=r.returncode, over_cap=r.over_cap, timed_out=False, peak_gib=r.peak_footprint_bytes / memcap.GIB)
            (raw / f"{name}.stderr.log").write_text(r.stderr)
        except subprocess.TimeoutExpired:
            row.update(exit=None, over_cap=False, timed_out=True, peak_gib=None)
    row.update(wall_s=w.wall_s, hygiene=w.problems)
    recs = [json.loads(l) for l in out.read_text().splitlines()] if out.exists() else []
    row["statuses"] = sorted({r["status"] for r in recs})
    off = next((r for r in recs if r["phase"] == "offline" and r["status"] == "ok"), None)
    on = next((r for r in recs if r["phase"] == "online" and r["status"] == "ok"), None)
    span = lambda m, k: (m.get(k) or 0) if not isinstance(m.get(k), list) else sum(m[k])
    if off:
        m = off["metrics"]
        row["offline_s"] = sum(span(m, k) for k in ("public_params_s", "server_preproc_s", "server_preproc_proof_s", "client_offline_s"))
        row["padding_factor"] = (off.get("scheme_params") or {}).get("padding_factor")
    if on:
        m = on["metrics"]
        row["query_s"] = sum(span(m, k) for k in ("query_gen_s", "server_answer_s", "client_recover_s", "client_verify_s"))
        row["correct"] = on["correct"]
    with open(raw / "feasibility.jsonl", "a") as f:
        f.write(json.dumps(row) + "\n")
    return row

{'run_id': 'grid-feasibility-20261111T064956Z', 'git_sha': '93ccbdac2f8da4d8505ec83dcf16fda473261b27', 'dirty': False}


In [2]:
# Every grid arm except S4 (measured in notebook 06) at the three sizes not yet run, smallest first, with the
# record-size extremes 32 B and 8 KiB. S3 retrieves one bit and takes no record size.
MiB = 1 << 20
cells = [(arm, db, rb) for db in (16 * MiB, 128 * MiB, 1024 * MiB)
         for arm in ("S0-go", "S0-cpp", "S1", "S2", "M0", "M1") for rb in (32, 8192)]
cells += [("S3", db, 0) for db in (16 * MiB, 128 * MiB, 1024 * MiB)]
cells.sort(key=lambda c: c[1])
rows = []
for arm, db, rb in cells:
    r = run_cell(arm, db, rb)
    rows.append(r)
    state = ("OVER 10 GiB CAP" if r["over_cap"] else "TIMED OUT (> 60 min)" if r["timed_out"]
             else f"exit {r['exit']}" + ("" if r["statuses"] == ["ok"] else f" {r['statuses']}"))
    print(f"{arm:6s} {db // MiB:5d} MiB {rb:5d} B  {state:22s} peak {r['peak_gib'] or 0:5.2f} GiB  "
          f"offline {r.get('offline_s', float('nan')):8.1f} s  query {r.get('query_s', float('nan')):7.2f} s  "
          f"wall {r['wall_s']:6.0f} s  {r['hygiene'] or ''}", flush=True)

S0-go     16 MiB    32 B  exit 0                 peak  0.15 GiB  offline      1.5 s  query    0.01 s  wall      3 s  


S0-go     16 MiB  8192 B  exit 0                 peak  0.19 GiB  offline      1.4 s  query    0.01 s  wall      2 s  


S0-cpp    16 MiB    32 B  exit 0                 peak  0.22 GiB  offline      4.3 s  query    0.01 s  wall      9 s  


S0-cpp    16 MiB  8192 B  exit 0                 peak  0.27 GiB  offline      5.7 s  query    0.01 s  wall     12 s  


S1        16 MiB    32 B  exit 0                 peak  0.55 GiB  offline     17.7 s  query    0.01 s  wall     36 s  


S1        16 MiB  8192 B  exit 0                 peak  0.67 GiB  offline     23.2 s  query    0.01 s  wall     47 s  


... (29 outputs left out)

M0      1024 MiB  8192 B  exit 0                 peak  2.01 GiB  offline      0.0 s  query    0.36 s  wall      1 s  


M1      1024 MiB    32 B  OVER 10 GiB CAP        peak 10.02 GiB  offline      nan s  query     nan s  wall     24 s  


M1      1024 MiB  8192 B  exit 0                 peak  5.14 GiB  offline      5.2 s  query    0.37 s  wall      6 s  


S3      1024 MiB     0 B  OVER 10 GiB CAP        peak 10.00 GiB  offline      nan s  query     nan s  wall   2249 s  


In [3]:
# Estimated grid cell times at the spec's repetition counts (spec 5): offline = (5 + 1 warm-up) repetitions, or
# (3 + 1) at 1 GiB; online = one untimed offline phase for setup, then (30 + 1) queries. Estimates from one
# measured repetition each, so they are bounds to plan with, not results.
for r in rows:
    if "offline_s" in r and "query_s" in r:
        off_reps = 4 if r["db_bytes"] == 1024 * MiB else 6
        r["est_offline_cell_min"] = off_reps * r["offline_s"] / 60
        r["est_online_cell_min"] = (r["offline_s"] + 31 * r["query_s"]) / 60
problems = [r for r in rows if r["over_cap"] or r["timed_out"] or r["exit"] != 0 or r["statuses"] != ["ok"] or r["hygiene"]]
print(f"{len(rows)} cells run; {len(problems)} did not finish cleanly:")
for r in problems:
    print(f"  {r['arm']} {r['db_bytes'] // MiB} MiB {r['record_bytes']} B: over_cap={r['over_cap']} timed_out={r['timed_out']} "
          f"exit={r['exit']} statuses={r['statuses']} hygiene={r['hygiene']}")
print("\nslowest estimated cells:")
for r in sorted((r for r in rows if "est_offline_cell_min" in r), key=lambda r: -(r["est_offline_cell_min"] + r["est_online_cell_min"]))[:10]:
    print(f"  {r['arm']:6s} {r['db_bytes'] // MiB:5d} MiB {r['record_bytes']:5d} B  offline cell {r['est_offline_cell_min']:7.1f} min  "
          f"online cell {r['est_online_cell_min']:7.1f} min  padding {r.get('padding_factor')}")
with open(raw / "estimates.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")

39 cells run; 7 did not finish cleanly:
  M1 128 MiB 32 B: over_cap=True timed_out=False exit=-9 statuses=[] hygiene=[]
  S1 1024 MiB 32 B: over_cap=True timed_out=False exit=-9 statuses=[] hygiene=[]
  S1 1024 MiB 8192 B: over_cap=True timed_out=False exit=-9 statuses=[] hygiene=[]
  S2 1024 MiB 32 B: over_cap=True timed_out=False exit=-9 statuses=[] hygiene=[]
  S2 1024 MiB 8192 B: over_cap=True timed_out=False exit=-9 statuses=[] hygiene=[]
  M1 1024 MiB 32 B: over_cap=True timed_out=False exit=-9 statuses=[] hygiene=[]
  S3 1024 MiB 0 B: over_cap=True timed_out=False exit=-9 statuses=[] hygiene=[]

slowest estimated cells:
  S2       128 MiB  8192 B  offline cell    19.3 min  online cell     3.2 min  padding 1
  S2       128 MiB    32 B  offline cell    19.3 min  online cell     3.2 min  padding 1
  S1       128 MiB  8192 B  offline cell    19.1 min  online cell     3.2 min  padding 1.4979511350393295
  S0-cpp  1024 MiB  8192 B  offline cell    17.4 min  online cell     4.5 min  pa